# Einsum Notation

See discussion on [Twitter](https://twitter.com/kyscg7/status/1679508714141937664), [GitHub Gists](https://gist.github.com/kyscg/fe6bfe5ddb0e34c918c06242f7979c87), [Mathstodon](https://mathstodon.xyz/@kyscg/110707391200337881).

---

This notebook has exercises to understand how Einstein Summation Notation works for Deep Learning algorithms. I initially tested my understanding by taking an operation (say matrix-vector multiplication), writing the corresponding einsum notation, and checking against the NumPy implementation using `all()`. I then moved on to PyTorch, and implemented an MLP class and a Multi-head Self-Attention block using einsum notation.

Sources: [A basic introduction to NumPy's einsum | ajcr.net](https://ajcr.net/Basic-guide-to-einsum/) by Alex Riley, and [Einsum is All you Need - Einstein Summation in Deep Learning | rockt.github.io](https://rockt.github.io/2018/04/30/einsum) by Tim Rocktäschel. The following are rules from Alex Riley's blog post which are a nice way to get introduced to the idea.

- Repeating letters between input arrays means that values along those axes will be multiplied together. The products make up the values for the output array.
- Omitting a letter from the output means that values along that axis will be summed.
- We can return the unsummed axes in any order we like.

I felt however, that a better way to understand einsum notation is to decompose them to loops. Say we have the following matrix multiplication:

```python
C = np.einsum("ij,jk -> ik", A, B)
```

This is essentially, the same as looping over `i` and `k`, and preparing sums. Then, loop over the variables that are not in the output, `j`, and sum over that variable. Which means, the loop looks like:

```python
for i in range(...):
    for k in range (...):
        C[i, k] = 0
        for j in range(...):
            C[i, k] += A[i, j] * B[j, k]
```

One example that bothered me was an einsum notation that returned the leading diagonal of the matrix, so I'll write it out here:

```python
diag = np.einsum("ii -> i", A)
```

Here, we loop once over `i`, and then directly add in only elements from the leading diagonal.

```python
for i in range(...):
    diag[i] = 0
    diag[i] += A[i, i]
```

This makes it easy to work out complex notation like batched matrix multiplication, inner products, tensor contractions etc. Note that if the right hand side of the notation is empty, we initialize the sum outside all the loops, and hence get the sum of the complete tensor.

### Log

- Other stuff to see, because it's too late in the night now, https://en.wikipedia.org/wiki/Einstein_notation, https://www.continuummechanics.org/tensornotationbasic.html
- I don't understand how `A[:, None]` is working, nor what it is doing.
    - Why is `A[: None].shape` giving me `(3, 1, 3)` when expected behavior is `(3, 3, 1)`? `A` is a 2D matrix of shape `(3, 3)`
    - More confusion. `A[:, None, None]` returns `(3, 1, 1, 3)` and `A[:, :, None, None]` returns `(3, 3, 1, 1)`.
- Amazing what a good night's sleep will do. I instantly realized that `A[:, None]` adds an empty second dimension because we've placed the `None` second. It is equivalent to `A[:, None, :]`.
    - If we want the shape to be `(3, 3, 1)`, we slice with the `None` in third place, like `A[:, :, None]`.
- https://stackoverflow.com/questions/tagged/numpy-einsum has many real world examples.
- [Tensor Contraction](https://en.wikipedia.org/wiki/Tensor_contraction) is the general form of Batch Matrix Multiplication.
- Bilinear transformation doesn't give the same result as einsum because PyTorch uses random weights to compute the transformation. I'm settling to check whether the shapes match.

In [1]:
import numpy as np

### Multiply `A` and `B` and sum along the rows

In [2]:
A = np.array([0, 1, 2]).reshape(3, 1)
B = np.array([[ 0,  1,  2,  3],
              [ 4,  5,  6,  7],
              [ 8,  9, 10, 11]])

In [3]:
print("Shape of A: ", A.shape)
print("Shape of B: ", B.shape)

Shape of A:  (3, 1)
Shape of B:  (3, 4)


In [4]:
(np.einsum("ij,ij -> i", A, B) == (A * B).sum(axis=1)).all()

True

### Matrix Multiplication


In [5]:
A = np.array([[1, 1, 1],
              [2, 2, 2],
              [5, 5, 5]])
B = np.array([[0, 1, 0],
              [1, 1, 0],
              [1, 1, 1]])

In [6]:
print("Shape of A: ", A.shape)
print("Shape of B: ", B.shape)

Shape of A:  (3, 3)
Shape of B:  (3, 3)


In [7]:
(np.einsum("ij,jk -> ik", A, B) == np.matmul(A, B)).all()

True

### Operations on 1D arrays

| Call signature | NumPy equivalent | Description |
|:---|:---:|---:|
| `('i', A)` | `A` | Returns a view of A |
| `('i->', A)` | `sum(A)` | Sums the values of A |
| `('i,i->i', A, B)` | `A * B` | Element-wise multiplication of A and B |
| `('i,i', A, B)` | `inner(A, B)` | Inner product of A and B |
| `('i,j->ij', A, B)` | `outer(A, B)` | Outer product of A and B |


In [8]:
A = np.array([1, 2, 3])
B = np.array([4, 5, 6])

In [9]:
print("Shape of A: ", A.shape)
print("Shape of B: ", B.shape)

Shape of A:  (3,)
Shape of B:  (3,)


In [10]:
(np.einsum("i -> i", A) == A)        .all()
(np.einsum("i ->  ", A) == np.sum(A)).all()

(np.einsum("i,i -> i ", A, B) == A * B)            .all()
(np.einsum("i,i ->   ", A, B) == np.matmul(A, B.T)).all()
(np.einsum("i,i ->   ", A, B) == np.inner(A, B))   .all()
(np.einsum("i,j -> ij", A, B) == np.outer(A, B))   .all()

True

### Operations on 2D arrays

| Call signature | NumPy equivalent | Description |
|:---|:---:|---:|
| `('ij', A)` | `A` | Returns a view of A |
| `('ji', A)` | `A.T` | View transpose of A |
| `('ii->i', A)` | `diag(A)` | View main diagonal of A |
| `('ii', A)` | `trace(A)` | Sums main diagonal of A |
| `('ij->', A)` | `sum(A)` | Sums the values of A |
| `('ij->j', A)` | `sum(A, axis=0)` | Sum down the columns of A (across rows) |
| `('ij->i', A)` | `sum(A, axis=1)` | Sum horizontally along the rows of A |
| `('ij,ij->ij', A, B)` | `A * B` | Element-wise multiplication of A and B |
| `('ij,ji->ij', A, B)` | `A * B.T` | Element-wise multiplication of A and B.T |
| `('ij,jk', A, B)` | `dot(A, B)` | Matrix multiplication of A and B |
| `('ij,kj->ik', A, B)` | `inner(A, B)` | Inner product of A and B |
| `('ij,kj->ikj', A, B)` | `A[:, None] * B` | Each row of A multiplied by B |
| `('ij,kl->ijkl', A, B)` | `A[:, :, None, None] * B` | Each value of A multiplied by B |


In [11]:
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
B = np.array([[10, 11, 12],
              [13, 14, 15],
              [16, 17, 18]])

In [12]:
print("Shape of A: ", A.shape)
print("Shape of B: ", B.shape)

Shape of A:  (3, 3)
Shape of B:  (3, 3)


In [13]:
(np.einsum("ij -> ij", A) == A)                .all()
(np.einsum("ji -> ij", A) == A.T)              .all()
(np.einsum("ii ->  i", A) == np.diag(A))       .all()
(np.einsum("ii ->   ", A) == np.trace(A))      .all()
(np.einsum("ij ->   ", A) == np.sum(A))        .all()
(np.einsum("ij ->  j", A) == np.sum(A, axis=0)).all()
(np.einsum("ij ->  i", A) == np.sum(A, axis=1)).all()

(np.einsum("ij,ij ->   ij", A, B) == A * B)                  .all()
(np.einsum("ij,ji ->   ij", A, B) == A * B.T)                .all()
(np.einsum("ij,jk ->   ik", A, B) == np.matmul(A, B))        .all()
(np.einsum("ij,jk ->     ", A, B) == np.sum(np.matmul(A, B))).all()
(np.einsum("ij,kj ->   ik", A, B) == np.matmul(A, B.T))      .all()
(np.einsum("ij,kj ->  ikj", A, B) == A[:, None, :] * B)      .all()
(np.einsum("ij,kl -> ijkl", A, B) == A[:, :, None, None] * B).all()

True

### PyTorch

In [14]:
import torch

In [15]:
a = torch.arange(6).reshape(2, 3)
b = torch.arange(3)
c = torch.arange(15).reshape(3, 5)
d = torch.arange(3, 6)
e = torch.arange(6, 12).reshape(2, 3)
f = torch.arange(3, 7)
g = torch.randn(3, 2, 5)
h = torch.randn(3, 5, 3)
i = torch.randn(2, 3, 5, 7)
j = torch.randn(11, 13, 3, 17, 5)
k = torch.randn(2, 3)
l = torch.randn(5, 3, 7)
m = torch.randn(2, 7)

In [16]:
(torch.einsum("ij -> ji", a) == a.T)                 .all()
(torch.einsum("ij ->   ", a) == torch.sum(a))        .all()
(torch.einsum("ij ->  j", a) == torch.sum(a, axis=0)).all()
(torch.einsum("ij ->  i", a) == torch.sum(a, axis=1)).all()

(torch.einsum("ij,j  ->  i", a, b) == torch.matmul(a, b)).all()
(torch.einsum("ij,jk -> ik", a, c) == torch.matmul(a, c)).all()
(torch.einsum("i,i   ->   ", b, d) == torch.dot(b, d))   .all()
(torch.einsum("ij,ij ->   ", a, e) == torch.sum(a * e))  .all()
(torch.einsum("ij,ij -> ij", a, e) == a * e)             .all()
(torch.einsum("i,j   -> ij", b, f) == torch.outer(b, f)) .all()

# Batched Matrix Multiplication
print((torch.einsum("bmn,bno -> bmo", g, h) == torch.bmm(g, h)).all().item())

# Tensor Contraction
print((torch.einsum("ijkl,mnjok -> ilmno", i, j) == torch.tensordot(i, j, dims=([1, 2], [2, 4]))).all().item())

# Bilinear Transformation
print((torch.einsum("ij,kjl,il -> ik", k, l, m).shape == torch.nn.Bilinear(k.shape[1], m.shape[1], 5, bias=False)(k, m).shape))

True
True
True


### Multi-layer Perceptron

An MLP class that uses `einsum` instead of `matmul`, that can be used to classify MNIST images. Note the intuitive notation that helps us label axes appropriately. Backpropagation works in the usual manner after loading data and choosing an optimizer and loss.

In [17]:
batch_size = 32
input_size = 784
hidden_size = 500
output_size = 10

class MLP(torch.nn.Module):
    def __init__(self):

        super().__init__()

        self.W = torch.nn.Parameter(torch.randn(input_size, hidden_size))
        self.b = torch.nn.Parameter(torch.randn(hidden_size))
        self.V = torch.nn.Parameter(torch.randn(hidden_size, output_size))
        self.c = torch.nn.Parameter(torch.randn(output_size))

    def forward(self, x):
        """
        h = sigmoid(Wx + b)
        y = softmax(Vh + c)
        """

        assert x.shape == (batch_size, input_size) # x: (batch_size, input_size)

        h = torch.sigmoid(torch.einsum("bi,ih -> bh", x, self.W) + self.b)
        y = torch.softmax(torch.einsum("bh,ho -> bo", h, self.V) + self.c,
                          dim=-1)
        return y

### Self-Attention

A class that implements the multi-head self attention algorithm to an input sequence. Note how we can skip all the calls to `transpose`, `contiguous`, and `reshape` by using einsum. Work out the multiplications by yourself to see how the notation helps writing code.

In [18]:
class SelfAttention(torch.nn.Module):
    def __init__(self, k, heads=8):
        """
        Initialize the MultiheadAttention layer.

        Args:
            k    : The embedding dimension.
            heads: The number of attention heads.

        Raises:
            ValueError: If k is not divisible by heads.
        """

        super().__init__()

        assert k % heads == 0, "k must be divisible by the number of heads"

        self.k = k
        self.heads = heads

        self.tokeys = torch.nn.Linear(k, k, bias=False)
        self.toqueries = torch.nn.Linear(k, k, bias=False)
        self.tovalues = torch.nn.Linear(k, k, bias=False)
        self.unifyheads = torch.nn.Linear(heads * (k // heads), k)

    def forward(self, x):
        """
        Applies multi-head attention to the input sequence.

        Args:
            x (torch.Tensor): The input sequence, of shape (b, t, k).

        Returns:
            torch.Tensor: The output sequence, of shape (b, t, k).

        Raises:
            ValueError: If the input embedding dimension does not match the layer embedding dimension.
        """

        b, t, k = x.size()
        h = self.heads
        assert k == self.k, f'Input embedding dimension={k} does not match layer embedding dimension={self.k}'

        # (b, t, k)
        queries = self.toqueries(x)
        keys    = self.tokeys(x)
        values  = self.tovalues(x)

        s = k // h

        # (b, t, k) --> (b, t, h, s)
        queries = queries.reshape(b, t, h, s)
        keys    = keys   .reshape(b, t, h, s)
        values  = values .reshape(b, t, h, s)

        # (b, h, t, t)
        dot = torch.einsum("bths,bdhs -> bhtd", (queries, keys)) / math.sqrt(s)
        dot = torch.nn.functional.softmax(dot, dim=3)

        # (b, t, h, s)
        out = torch.einsum("bhtl,blhs -> bths", (dot, values))

        # (b, t, k)
        out = out.reshape(b, t, h * s)
        return self.unifyheads(out)

### Tests

This is a later addition because I messed up the original MLP implementation (see [this comment](https://gist.github.com/kyscg/fe6bfe5ddb0e34c918c06242f7979c87?permalink_comment_id=5384223#gistcomment-5384223)). I am going to write some tests to check if the other stuff is working correctly.

In [19]:
batch_size = 32
input_size = 784
hidden_size = 500
output_size = 10

x = torch.randn(batch_size, input_size)
W = torch.randn(input_size, hidden_size)
b = torch.randn(hidden_size)

print((x @ W + b == torch.einsum('bi,ih,h -> bh', x, W, b)).all().item()) # incorrect
print((x @ W + b == torch.einsum('bi,ih -> bh', x, W) + b).all().item())


False
True


In [20]:
b = 32
t = 10
h = 8
s = 2

queries = torch.randn(b, t, h, s)
keys = torch.randn(b, t, h, s)

eins = torch.einsum("bths,bdhs -> bhtd", (queries, keys))
matm = torch.matmul(queries.transpose(2, 1), keys.transpose(2, 1).transpose(3, 2))

print(eins.shape == matm.shape)
print((eins == matm).all().item())


True
True
